# F1-Score Optimization & Feature Selection (Tree Models Only)

**Obiettivo:** Massimizzare la metrica **F1-Score** per la classificazione degli incidenti.

**Strategia:**
1.  **Dati:** Utilizzo del dataset `processed_v4_hybrid` (già bilanciato con SMOTE/Undersampling nel training).
2.  **Feature Selection:** Selezione rigorosa delle **Top 15 features** per ridurre il rumore.
3.  **Modelli Neutri:** Rimozione di tutti i parametri di bilanciamento forzato (`class_weight`, `scale_pos_weight`) dato che i dati di training sono già 50/50.
4.  **Threshold Tuning:** Ottimizzazione dinamica della soglia di decisione (invece del default 0.5) per trovare il punto di massimo F1-Score.

**Modelli:** XGBoost, Random Forest, Decision Tree.

## 1. Setup e Caricamento Dati

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import os
import json

# PyTorch per MLP
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline
sns.set_style('whitegrid')

# Device setup (MPS per MacBook Pro)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Utilizzo MPS (Metal Performance Shaders)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Utilizzo CUDA")
else:
    device = torch.device("cpu")
    print("Utilizzo CPU")

print("Librerie importate.")

In [ ]:
# Caricamento dataset v4 (Hybrid)
print("Caricamento dataset processed_v4_hybrid...")
X_train = pd.read_csv('../data/processed_v4_hybrid/X_train.csv')
y_train = pd.read_csv('../data/processed_v4_hybrid/y_train.csv')['BinaryIncidentGrade']
X_test = pd.read_csv('../data/processed_v4_hybrid/X_test.csv')
y_test = pd.read_csv('../data/processed_v4_hybrid/y_test.csv')['BinaryIncidentGrade']

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# Verifica bilanciamento training
print("\nDistribuzione classi Training (dovrebbe essere 50/50):")
print(y_train.value_counts(normalize=True))

## 2. Feature Selection (Top 15)
Utilizziamo un Random Forest veloce per identificare le 15 feature più discriminanti e scartare il resto.

In [ ]:
print("Esecuzione Feature Selection (Random Forest)...")

# RF leggero per selezione
selector = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
selector.fit(X_train, y_train)

# Estrazione importanza
importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': selector.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Selezione Top 15
top_15_features = importances.head(15)['Feature'].tolist()

print("\nTop 15 Features selezionate:")
for i, f in enumerate(top_15_features, 1):
    print(f"{i}. {f} ({importances[importances['Feature']==f]['Importance'].values[0]:.4f})")

# Filtraggio dataset
X_train_sel = X_train[top_15_features].copy()
X_test_sel = X_test[top_15_features].copy()

print(f"\nNuova shape X_train: {X_train_sel.shape}")

## 3. Definizione Modelli (No Class Weights)
Definiamo i modelli rimuovendo i parametri di bilanciamento, dato che il training set è già bilanciato.

In [ ]:
# Dizionario per i risultati
results = {}

# 1. XGBoost (scale_pos_weight rimosso)
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
    # scale_pos_weight RIMOSSO
)

# 2. Random Forest (class_weight RIMOSSO)
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
    # class_weight RIMOSSO
)

# 3. Decision Tree (class_weight RIMOSSO)
dt_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=20,
    random_state=42
    # class_weight RIMOSSO
)

# 4. MLP (PyTorch) - Architettura da 8-MLP.ipynb
class MLP(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(dropout),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(dropout),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(dropout),
            
            nn.Linear(32, 1)
            # No Sigmoid here - useremo BCEWithLogitsLoss
        )
    
    def forward(self, x):
        return self.network(x)

models_dict = {
    'XGBoost': xgb_model,
    'RandomForest': rf_model,
    'DecisionTree': dt_model,
    # MLP verrà gestito separatamente per via del training PyTorch
}

print("Modelli configurati: XGBoost, RandomForest, DecisionTree, MLP (PyTorch).")

## 4. Training e Threshold Optimization Loop
Per ogni modello:
1. Train
2. Predict Proba (Test Set)
3. Trova soglia che massimizza F1

In [ ]:
def optimize_threshold(y_true, y_probs, model_name):
    best_th = 0.5
    best_f1 = 0
    best_metrics = {}
    
    # Cerca la soglia migliore tra 0.1 e 0.95
    thresholds = np.arange(0.1, 0.95, 0.01)
    f1_scores = []
    
    for th in thresholds:
        preds = (y_probs >= th).astype(int)
        score = f1_score(y_true, preds)
        f1_scores.append(score)
        
        if score > best_f1:
            best_f1 = score
            best_th = th
            best_metrics = {
                'Precision': precision_score(y_true, preds),
                'Recall': recall_score(y_true, preds),
                'Accuracy': (preds == y_true).mean()
            }
    
    # Plot andamento F1 al variare della soglia
    plt.figure(figsize=(8, 4))
    plt.plot(thresholds, f1_scores, label='F1 Score')
    plt.axvline(best_th, color='r', linestyle='--', label=f'Best Th: {best_th:.2f}')
    plt.title(f'F1 Score vs Threshold - {model_name}')
    plt.xlabel('Threshold')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.show()
    
    return best_th, best_f1, best_metrics

# --- TRAIN TREE MODELS --- 
for name, model in models_dict.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_sel, y_train)
    
    # Get probabilities (classe 1)
    probs = model.predict_proba(X_test_sel)[:, 1]
    
    # Optimize
    best_th, best_f1, metrics = optimize_threshold(y_test, probs, name)
    
    results[name] = {
        'Best Threshold': best_th,
        'F1 Score': best_f1,
        **metrics,
        'ROC AUC': roc_auc_score(y_test, probs)
    }
    print(f"  -> Best Threshold: {best_th:.2f}")
    print(f"  -> Max F1 Score: {best_f1:.4f}")

## 4.1 Training MLP (PyTorch)
Il training MLP richiede preprocessing separato (StandardScaler) e gestione del device (MPS/CUDA/CPU).

In [ ]:
# --- Funzioni MLP (da 8-MLP.ipynb) ---
def train_epoch_mlp(model, loader, criterion, optimizer, device):
    """Training di una singola epoca."""
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_mlp(model, loader, device):
    """Valutazione modello. Restituisce labels, predictions, probabilities."""
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())
    
    return np.array(all_labels).flatten(), np.array(all_preds).flatten(), np.array(all_probs).flatten()

print("Funzioni MLP definite ✅")

In [ ]:
# --- PREPARAZIONE DATI MLP ---
print("="*50)
print("Preparazione dati per MLP...")
print("="*50)

# Standardizzazione (critica per neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

# Conversione a tensori PyTorch
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

print(f"\nTensori creati:")
print(f"  X_train_tensor: {X_train_tensor.shape}, dtype: {X_train_tensor.dtype}")
print(f"  y_train_tensor: {y_train_tensor.shape}, dtype: {y_train_tensor.dtype}")

# DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

print(f"\nDataLoaders creati:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")
print("\n✅ Dati pronti per MLP")

In [ ]:
# --- INIZIALIZZAZIONE MODELLO MLP ---
print("="*50)
print("Inizializzazione modello MLP...")
print("="*50)

# Dimensione input
input_dim = X_train_sel.shape[1]
print(f"Input dimension: {input_dim}")

# Inizializza modello e sposta su device
mlp_model = MLP(input_dim=input_dim, dropout=0.3).to(device)

# Loss senza class weights (dati già bilanciati)
criterion = nn.BCEWithLogitsLoss()
mlp_optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

print(f"\nDevice: {device}")
print(f"Architettura: {input_dim} → 128 → 64 → 32 → 1")
print(f"Parametri totali: {sum(p.numel() for p in mlp_model.parameters()):,}")
print(f"Learning rate: 0.001")
print(f"Dropout: 0.3")
print("\n✅ Modello MLP inizializzato")

In [ ]:
# --- TRAINING MLP ---
print("="*50)
print("Training MLP...")
print("="*50)

n_epochs = 50
train_losses = []

for epoch in range(n_epochs):
    train_loss = train_epoch_mlp(mlp_model, train_loader, criterion, mlp_optimizer, device)
    train_losses.append(train_loss)
    
    # Stampa ogni 5 epoche
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/{n_epochs} - Loss: {train_loss:.4f}")

print("\n✅ Training MLP completato!")

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(range(1, n_epochs+1), train_losses, marker='o', markersize=3, color='blue')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('BCE Loss', fontsize=12)
plt.title('MLP Training Loss', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- VALUTAZIONE E THRESHOLD OPTIMIZATION MLP ---
print("="*50)
print("Valutazione MLP e Threshold Optimization...")
print("="*50)

# Predizioni sul test set
y_true_mlp, y_pred_mlp, mlp_probs = evaluate_mlp(mlp_model, test_loader, device)

print(f"Predizioni MLP:")
print(f"  y_true shape: {y_true_mlp.shape}")
print(f"  probs range: [{mlp_probs.min():.4f}, {mlp_probs.max():.4f}]")

# Threshold optimization
best_th_mlp, best_f1_mlp, metrics_mlp = optimize_threshold(y_test.values, mlp_probs, 'MLP')

# Aggiungi risultati MLP al dizionario
results['MLP'] = {
    'Best Threshold': best_th_mlp,
    'F1 Score': best_f1_mlp,
    **metrics_mlp,
    'ROC AUC': roc_auc_score(y_test, mlp_probs)
}

print(f"\n{'='*50}")
print("RISULTATI MLP:")
print(f"{'='*50}")
print(f"  Best Threshold: {best_th_mlp:.2f}")
print(f"  Max F1 Score:   {best_f1_mlp:.4f}")
print(f"  Precision:      {metrics_mlp['Precision']:.4f}")
print(f"  Recall:         {metrics_mlp['Recall']:.4f}")
print(f"  ROC AUC:        {results['MLP']['ROC AUC']:.4f}")

## 5. Confronto Finale e Risultati

In [ ]:
results_df = pd.DataFrame(results).T.sort_values(by='F1 Score', ascending=False)

print("\n=== CLASSIFICA FINALE (Ordinata per F1-Score) ===")
print(results_df[['F1 Score', 'Best Threshold', 'Precision', 'Recall', 'Accuracy', 'ROC AUC']].round(4))

# Visualizzazione F1 vs Threshold per tutti
plt.figure(figsize=(10, 6))
sns.barplot(x=results_df.index, y=results_df['F1 Score'], palette='viridis')
plt.ylim(0.5, 0.9) # Zoom sulla parte alta
plt.title('Confronto F1 Score Ottimizzato')
plt.ylabel('Max F1 Score')
plt.show()

In [ ]:
# Analisi Features Utilizzate
print("Features utilizzate nel training (Top 15):")
print(top_15_features)